In [1]:
import os
import tensorflow as tf
import xml.etree.ElementTree as ET
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.utils import shuffle

In [2]:
import os

base_dir = r"C:\Users\Andhavarapu Jahnavi\Desktop\smart-survellience-system\dataset\train"

for folder in os.listdir(base_dir):
    path = os.path.join(base_dir, folder)
    if os.path.isdir(path):
        print(folder, "->", len(os.listdir(path)), "images")


knife -> 400 images
not_knife -> 622 images


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import os

base_dir = r"C:\Users\Andhavarapu Jahnavi\Desktop\smart-survellience-system\dataset\train"

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=False
)

train_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training',
    shuffle=True
)

validation_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

print("Class indices:", train_generator.class_indices)
# EXPECTED: {'knife': 0, 'not_knife': 1}

model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),              # 🔥 IMPORTANT
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator,
    verbose=1
)

model.save("knife_classifier_model.h5")


Found 818 images belonging to 2 classes.
Found 204 images belonging to 2 classes.


Class indices: {'knife': 0, 'not_knife': 1}


c:\Users\Andhavarapu Jahnavi\Desktop\smart-survellience-system\fr_venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 25s 895ms/step - accuracy: 0.6027 - loss: 0.6641 - val_accuracy: 0.6078 - val_loss: 0.6392
Epoch 2/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 18s 691ms/step - accuracy: 0.6112 - loss: 0.6181 - val_accuracy: 0.6569 - val_loss: 0.6297
Epoch 3/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 18s 679ms/step - accuracy: 0.6540 - loss: 0.5937 - val_accuracy: 0.7304 - val_loss: 0.5606
Epoch 4/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 19s 730ms/step - accuracy: 0.7237 - loss: 0.5349 - val_accuracy: 0.7010 - val_loss: 0.5328
Epoch 5/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 18s 674ms/step - accuracy: 0.7861 - loss: 0.4561 - val_accuracy: 0.8235 - val_loss: 0.4422
Epoch 6/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 18s 673ms/step - accuracy: 0.8227 - loss: 0.4110 - val_accuracy: 0.7843 - val_loss: 0.4554
Epoch 7/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 22s 872ms/step - accuracy: 0.8337 - loss: 0.3879 - val_accuracy: 0.8676 - val_loss: 0.3438
Epoch 8/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 25s 978ms/step - accuracy: 0.8545 - loss: 0.3635 - val_accu

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model


# Path to test dataset
test_dir = r"C:\Users\Andhavarapu Jahnavi\Desktop\smart-survellience-system\dataset\test"

# Test data generator (NO augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

# Evaluate model on test data
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)

print(f"\nTest Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")


Found 528 images belonging to 2 classes.
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 139ms/step - accuracy: 0.9053 - loss: 0.2310

Test Accuracy: 90.53%
Test Loss: 0.2310


In [5]:
model.save("knife_classifier_model.h5")
print("Model saved as knife_classifier_model.h5")

Model saved as knife_classifier_model.h5


In [10]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array  
from tensorflow.keras.models import load_model  

test_dir = r"C:\Users\Andhavarapu Jahnavi\Desktop\smart-survellience-system\dataset\test"

def predict_image(image_path, model):
    img = load_img(image_path, target_size=(224, 224))  
    img_array = img_to_array(img) / 255.0  
    img_array = np.expand_dims(img_array, axis=0)  

    prediction = model.predict(img_array)[0][0]  
    return prediction  
for filename in os.listdir(test_dir):
    file_path = os.path.join(test_dir, filename)
    if os.path.isfile(file_path):  
        
        prediction = predict_image(file_path, model)

        if prediction > 0.5:
            print(f"{filename}: Not Knife (Confidence: {prediction:.2f})")
        else:
            print(f"{filename}: Knife (Confidence: {1 - prediction:.2f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
image7537.jpg: Not Knife (Confidence: 0.98)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
image7539.jpg: Not Knife (Confidence: 1.00)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
image7541.jpg: Not Knife (Confidence: 1.00)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
image7547.jpg: Not Knife (Confidence: 0.95)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
image7549.jpg: Not Knife (Confidence: 1.00)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
image755.jpg: Not Knife (Confidence: 0.99)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
image7550.jpg: Not Knife (Confidence: 1.00)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
image7557.jpg: Not Knife (Confidence: 0.68)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
image7558.jpg: Not Knife (Confidence: 1.00)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
image756.jpg: Not Knife (Confidence: 1.00)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
image7562.jpg: Not Knife (Confidence: 0.95)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
image7564.jpg: Not Knife (Confidence: 0.82)
1/1 ━━━━━━━━━━━━━━